# データ保存

In [ ]:
import os
os.environ["SPEDAS_DATA_DIR"] = "/mnt/j/observation_data/"

# 電場のデータを弄ってみる

In [ ]:
import pyspedas as psp
import pytplot as pt
import numpy as np
import xarray as xr
import pandas as pd

pt.del_data('*')

time_range = ['20220901/21:00:00', '20220902/00:00:00']
pt.timespan('2022-09-01/22:25:00', 1, keyword='hour')

path_base_save_plot = f'/mnt/j/KAW_observation/E_B_ratio_ERG/2022-09-01/2230-2330_avg_3numden'

psp.erg.pwe_efd(trange=time_range, level='l2', datatype='64', coord='dsi', no_update=True)
psp.erg.mgf(trange=time_range, level='l2', datatype='64hz', coord='dsi', no_update=True)

In [ ]:
E64_data_Ex = pt.data_quants['erg_pwe_efd_l2_E64Hz_dsi_Ex_waveform']
E64_data_Ey = pt.data_quants['erg_pwe_efd_l2_E64Hz_dsi_Ey_waveform']
B64_data    = pt.data_quants['erg_mgf_l2_mag_64hz_dsi']

time_range_T = [time_range[0].replace('/', 'T'), time_range[1].replace('/', 'T')]
E64_data_Ex = E64_data_Ex.sortby('time').sel(time=slice(time_range_T[0], time_range_T[1]))
E64_data_Ey = E64_data_Ey.sortby('time').sel(time=slice(time_range_T[0], time_range_T[1]))
B64_data    = B64_data.sortby('time').sel(time=slice(time_range_T[0], time_range_T[1]))

print(E64_data_Ex)
print(E64_data_Ey)
print(B64_data)

In [ ]:
import xarray as xr
import numpy as np

# --- 0. 準備：変数名は質問に合わせている -----------------------------
Ex = E64_data_Ex.sortby('time')           # 時系列を昇順に
Ey = E64_data_Ey.sortby('time')
B  = B64_data.sortby('time')              # 64 Hz 磁場ベクトル

Bx = B.isel(v_dim=0)         # dims = ('time',)
By = B.isel(v_dim=1)
Bz = B.isel(v_dim=2)

# --- 1. E を B のタイムスタンプへ線形補間 ----------------------------
Ex_i = Ex.interp(time=B.time, method='linear')
Ey_i = Ey.interp(time=B.time, method='linear')
#   • 範囲外はデフォルトで NaN。必要なら
#     .interp(..., kwargs={'fill_value': 'extrapolate'}) など

# --- 2. Ez を計算 ----------------------------------------------------
EPS = 1e-12          # ゼロ割り回避用の閾値 (単位: nT)
Ez = xr.where(np.abs(Bz) > EPS, -(Ex_i*Bx + Ey_i*By)/Bz, np.nan)
Ez.name = 'E64Hz_dsi_Ez_waveform'

# --- 3. 必要なら Dataset にまとめる --------------------------------
ds = xr.Dataset({
    'Ex_dsi': Ex_i,
    'Ey_dsi': Ey_i,
    'Ez_dsi': Ez,
    'Bx_dsi': ('time', Bx.data),
    'By_dsi': ('time', By.data),
    'Bz_dsi': ('time', Bz.data),
}, coords={'time': B.time})

ds = ds.dropna(dim='time', how='any', subset=['Ex_dsi', 'Ey_dsi', 'Bx_dsi', 'By_dsi', 'Bz_dsi'])

ds

In [ ]:
import pytplot as pt
import os
import matplotlib.pyplot as plt

# ---- xarray → tplot へ格納 ----------------------------
for var in ['Ex_dsi', 'Ey_dsi', 'Ez_dsi', 'Bx_dsi', 'By_dsi', 'Bz_dsi']:
    pt.store_data(
        var,
        data={'x': ds['time'].values, 'y': ds[var].values},
    )

pt.options(['Ex_dsi', 'Ey_dsi', 'Ez_dsi'], 'ysubtitle', '[mV/m]')
pt.options('Ex_dsi', 'ytitle', 'E_x (DSI)')
pt.options('Ey_dsi', 'ytitle', 'E_y (DSI)')
pt.options('Ez_dsi', 'ytitle', 'E_z (DSI)')
pt.options(['Bx_dsi', 'By_dsi', 'Bz_dsi'], 'ysubtitle', '[nT]')
pt.options('Bx_dsi', 'ytitle', 'B_x (DSI)')
pt.options('By_dsi', 'ytitle', 'B_y (DSI)')
pt.options('Bz_dsi', 'ytitle', 'B_z (DSI)')

# ---- プロット -----------------------------------------
# 電場 3 成分 + 磁場 3 成分を上下 2 パネルに分けて描画
vars_to_plot = ['Ex_dsi', 'Ey_dsi', 'Ez_dsi', 'Bx_dsi', 'By_dsi', 'Bz_dsi']
if os.path.isdir(path_base_save_plot):
    # フォルダが存在 → プロットを「表示せずに保存」
    fig, axes = pt.tplot(
        vars_to_plot,
        display=False,              # Jupyter 上での自動表示を抑制
        return_plot_objects=True,   # fig, axes を返してもらう
        save_png=os.path.join(path_base_save_plot, 'EB_fields_dsi.png')
    )
    plt.close(fig)
else:
    # フォルダが存在しない → プロットを通常表示
    pt.tplot(
        vars_to_plot,
        display=True   # 省略可（デフォルト）
    )

In [ ]:
import xarray as xr
import numpy as np

def dsi_to_fac(ds: xr.Dataset, window_sec: float = 100.0) -> xr.Dataset:
    """DSI→FAC 変換 (64 Hz データを想定)
    Parameters
    ----------
    ds : xr.Dataset
        必須変数: Ex_dsi, Ey_dsi, Ez_dsi, Bx_dsi, By_dsi, Bz_dsi
    window_sec : float, optional
        FAC 基底を決める移動平均時間 [s]
    Returns
    -------
    ds_out : xr.Dataset
        DSI + FAC の両方を含む Dataset
    """
    # --- 0. 時系列を昇順にしておく -----------------------------------
    ds = ds.sortby('time')

    # --- 1. 平均磁場 <B> の計算 --------------------------------------
    dt = (ds.time[1] - ds.time[0]).astype('timedelta64[ns]').astype(float) * 1e-9
    win_pts = int(window_sec / dt)
    B_dsi = ds[['Bx_dsi', 'By_dsi', 'Bz_dsi']].to_array('comp')      # (comp,time)
    print(B_dsi)

    B_avg = B_dsi.rolling(time=win_pts, center=True).mean('time')          # (comp,time)
    B_avg = B_avg.transpose('time', 'comp').values                   # (N,3)

    # --- 2. FAC 基底ベクトル -----------------------------------------
    z0 = np.array([0.0, 0.0, 1.0])
    e2 = np.cross(z0, B_avg)                                              # ⟂B & ⟂z
    # z0 // B のとき e2≈0 → 代わりに x軸とクロスする等の処理を追加しても良い
    e1 = np.cross(e2, B_avg)
    e3 = B_avg

    def _normalize(v):
        return v / np.linalg.norm(v, axis=1, keepdims=True)

    e1_hat = _normalize(e1)
    e2_hat = _normalize(e2)
    e3_hat = _normalize(e3)

    # --- 3. 回転行列 R (time,3,3) とベクトル変換 ----------------------
    R = np.stack([e1_hat, e2_hat, e3_hat], axis=2)                   # columns=ê_i

    # DSI→FAC: v_fac = R · v_dsi
    E_dsi = ds[['Ex_dsi', 'Ey_dsi', 'Ez_dsi']].to_array('comp') \
              .transpose('time', 'comp').values                      # (N,3)
    B_dsi_np = B_dsi.transpose('time', 'comp').values               # (N,3)

    E_fac = np.einsum('tji,tj->ti', R, E_dsi)                        # (N,3)
    B_fac = np.einsum('tji,tj->ti', R, B_dsi_np)                     # (N,3)

    # --- 4. Dataset へ格納 ------------------------------------------
    ds_out = ds.copy()
    ds_out['Ex_fac'], ds_out['Ey_fac'], ds_out['Ez_fac'] = [
        (('time'), E_fac[:, k]) for k in range(3)]
    ds_out['Bx_fac'], ds_out['By_fac'], ds_out['Bz_fac'] = [
        (('time'), B_fac[:, k]) for k in range(3)]

    # attrs など必要ならここで設定
    return ds_out, R, e3_hat


# ---------- 使い方 -------------------------------------------------------
ds_fac, Rotation_tensor, e3_hat = dsi_to_fac(ds)

print(ds_fac[['Ex_fac', 'Ey_fac', 'Ez_fac']])
print(ds_fac[['Bx_fac', 'By_fac', 'Bz_fac']])

print(Rotation_tensor[150000, :, :])
print(e3_hat[150000, :])

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt


# --- 角度 θ(t) = arccos( ê₃ · ẑ_DSI ) -----------------------------
e3_z     = Rotation_tensor[:, 2, 2]                  # (N,) ← 第3列・z成分
angle_deg = np.degrees(np.arccos(e3_z))              # 0–180° に丸め
t         = ds_fac.time.values                       # or ds.time.values

# フォルダが存在するか
if os.path.isdir(path_base_save_plot):
    # ── 存在する → プロットを作って保存のみ、Notebook上には表示しない
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(t, angle_deg, lw=1)
    ax.set_ylabel(r'∠($\bf{B}_{0}$, $\bf{\hat{z}}$(DSI))  [deg]')
    ax.set_xlabel('Time')
    ax.set_title(r'Rotation angle between $\bf{B}_{0}$ and ERG spin axis')
    ax.grid(True)
    plt.tight_layout()

    # ファイル名は適宜変更
    save_path = os.path.join(path_base_save_plot, 'rotation_angle.png')
    fig.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close(fig)

else:
    # ── 存在しない → 通常どおり表示だけ
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(t, angle_deg, lw=1)
    ax.set_ylabel(r'∠($\bf{B}_{0}$, $\bf{\hat{z}}$(DSI))  [deg]')
    ax.set_xlabel('Time')
    ax.set_title(r'Rotation angle between $\bf{B}_{0}$ and ERG spin axis')
    ax.grid(True)
    plt.tight_layout()
    plt.show()


In [ ]:
import os
import pytplot as pt
import numpy as np

# --- 1. pytplot 変数への格納 はそのまま ----
epoch = ds_fac.time.values.astype('datetime64[ns]').astype('int64') * 1e-9

for comp in ['x', 'y', 'z']:
    pt.store_data(f'E_fac_{comp}', data={'x': epoch, 'y': ds_fac[f'E{comp}_fac'].values})
    pt.store_data(f'B_fac_{comp}', data={'x': epoch, 'y': ds_fac[f'B{comp}_fac'].values})

    pt.options(f'E_fac_{comp}', 'ytitle', f'E_{comp} (FAC)\n[mV/m]')
    pt.options(f'B_fac_{comp}', 'ytitle', f'B_{comp} (FAC)\n[nT]')

for comp in ['perp', 'para']:
    if comp == 'perp':
        ds_E = np.sqrt(ds_fac['Ex_fac'].values**2 + ds_fac['Ey_fac'].values**2)
        ds_B = np.sqrt(ds_fac['Bx_fac'].values**2 + ds_fac['By_fac'].values**2)
    else:  # 'para'
        ds_E = ds_fac['Ez_fac'].values
        ds_B = ds_fac['Bz_fac'].values

    pt.store_data(f'E_fac_{comp}', data={'x': epoch, 'y': ds_E})
    pt.store_data(f'B_fac_{comp}', data={'x': epoch, 'y': ds_B})

    pt.options(f'E_fac_{comp}', 'ytitle', f'E_{comp} (FAC)\n[mV/m]')
    pt.options(f'B_fac_{comp}', 'ytitle', f'B_{comp} (FAC)\n[nT]')

# ------------------------------------------------------------
# 2. フォルダ存在チェック付きで tplot 発行
# ------------------------------------------------------------
vars_to_plot = [
    'E_fac_x', 'E_fac_y', 'E_fac_z',
    'B_fac_x', 'B_fac_y', 'B_fac_z'
]

if os.path.isdir(path_base_save_plot):
    # フォルダがある → 表示せずに保存のみ
    fig, axes = pt.tplot(
        vars_to_plot,
        display=False,              # Jupyter 上の自動表示を抑制
        return_plot_objects=True,   # fig, axes を返してもらう
        save_png=os.path.join(path_base_save_plot, 'EB_fields_fac.png')
    )
    # 完全に表示を防ぐために閉じる
    import matplotlib.pyplot as plt
    plt.close(fig)

else:
    # フォルダがない → 通常の表示のみ
    pt.tplot(
        vars_to_plot,
        display=True   # デフォルトなので省略可
    )

## Poynting fluxのplot

In [ ]:
mu_0 = 4e-7 * np.pi  # [H/m] = [N/A^2]

# parallel Poynting flux S_parallel = (E_fac_x*B_fac_y - E_fac_y*B_fac_x)/mu_0
E_fac_x = pt.data_quants['E_fac_x'] * 1e-3  # [mV/m] → [V/m]
E_fac_y = pt.data_quants['E_fac_y'] * 1e-3
B_fac_x = pt.data_quants['B_fac_x'] * 1e-9  # [nT] → [T]
B_fac_y = pt.data_quants['B_fac_y'] * 1e-9
S_parallel = (E_fac_x*B_fac_y - E_fac_y*B_fac_x)/mu_0 * 1e6  # [W/m^2] → [μW/m^2]
pt.store_data('S_parallel', data={'x': E_fac_x.time, 'y': S_parallel})
pt.options('S_parallel', 'ytitle', r'$S_{\parallel}$')
pt.options('S_parallel', 'ysubtitle', r'[$\mathrm{\mu W/m^{2}}$]')

vars_to_plot = ['E_fac_x', 'E_fac_y', 'B_fac_x', 'B_fac_y', 'S_parallel']

pt.tplot(vars_to_plot, display=True)

In [ ]:
import os
import numpy as np
from scipy.signal import butter, sosfiltfilt
import pytplot as pt
import matplotlib.pyplot as plt


# ---------- 1. Butterworth ハイパス設定 -----------------------
fs     = 64.0              # サンプリング周波数 [Hz]
cutoff = 1/6               # 0.25 Hz (spin 周期 8 s より少し高め)
sos    = butter(N=4, Wn=cutoff / (fs/2),
                btype='high', output='sos')

fs = 64.0                  # サンプリング周波数 [Hz]
order = 4                   # 4次 Butterworth
spin_tone = 1.0/8.0           # 減衰させたいスピン周期 [Hz]

# 除去したい周波数帯域
lowcut = spin_tone - 0.02
highcut = spin_tone*4 + 0.1

# フィルタ係数を計算
sos = butter(N=order, Wn=[lowcut, highcut], btype='bandstop', output='sos', fs=fs)

def apply_filter_segmented(y, sos_mat):
    """NaN を含む 1‑D 配列にセグメントごとで sosfiltfilt を適用する"""
    good = np.isfinite(y)
    out  = np.full_like(y, np.nan)
    idx  = np.where(good)[0]
    segs = np.split(idx, np.where(np.diff(idx) != 1)[0] + 1)
    
    # パディング長はフィルタの次数に依存
    # scipyのドキュメントによると、sosfiltfiltのデフォルトpadlenは 3 * (sos.shape[1] // 2 - 1)
    # sosの形状は (n_sections, 6) なので、padlenは 3 * 2 = 6 となる
    padlen = 3 * (sos_mat.shape[1] - 1)
    
    for s in segs:
        if s.size > padlen:
            out[s] = sosfiltfilt(sos_mat, y[s])
    return out

# ---------- 2. B_fac_* にフィルタを掛けて新変数を登録 ---------------
epoch = ds_fac.time.values.astype('datetime64[ns]').astype('int64') * 1e-9

for comp in ['x', 'y', 'z']:
    for EB in ['E', 'B']:
        var_in  = f'{EB}_fac_{comp}'
        da      = pt.data_quants[var_in]
        BS_data = da.values#apply_filter_segmented(da.values, sos) # Filterを適用しない
        var_out = f'{var_in}_BS'
        pt.store_data(var_out,
                      data={'x': da.time.values, 'y': BS_data})
        pt.options(var_out, 'ytitle',        f'{EB}_{comp} (BS)\n[nT]' if EB=='B' else f'{EB}_{comp} (BS)\n[mV/m]')

# ---------- 3. プロット（フォルダある→保存のみ／ない→表示） ----
vars_BS = [
    'E_fac_x_BS', 'E_fac_y_BS', 'E_fac_z_BS',
    'B_fac_x_BS', 'B_fac_y_BS', 'B_fac_z_BS'
]

if os.path.isdir(path_base_save_plot):
    # 存在する → 表示せずに保存のみ
    fig, axes = pt.tplot(
        vars_BS,
        display=False,
        return_plot_objects=True,
        save_png=os.path.join(path_base_save_plot, 'EB_fields_fac_BS.png')
    )
    plt.close(fig)
else:
    # 存在しない → 通常表示
    pt.tplot(
        vars_BS,
        display=True
    )


In [ ]:
import os
import sys
import importlib
import numpy as np
import matplotlib.pyplot as plt
import pyspedas as psp
import pytplot as pt

sys.path.append("..")
import module_handmade.tdwavelet as tw
importlib.reload(tw)

# 存在チェック用にフォルダを作らない派ならコメントアウトしてOK
# os.makedirs(path_base_save_plot, exist_ok=True)

# 1. 各成分に対して CWT 計算とオプション設定
for EB, unit in [('B', 'nT^2/Hz'), ('E', '(mV/m)^2/Hz')]:
    for comp in ['x', 'y', 'z']:
        var_in = f'{EB}_fac_{comp}_BS'
        if var_in not in pt.data_quants:
            print(f'skip (no {var_in})')
            continue

        tw.tdwavelet(
            var_in,
            dt=1/64,
            s0=1/64*4,
            dj=1/16,
            suffix='_cwt',
            zrange=[1e-6, 1e2]
        )

        var_out = f'{var_in}_cwt'
        # 軸設定
        pt.options(var_out, 'ylog', 1)
        pt.options(var_out, 'ztitle', 'PSD')
        pt.options(var_out, 'zsubtitle', unit)
        pt.options(var_out, 'ytitle', f'{EB}_{comp} (FAC)')
        pt.options(var_out, 'ysubtitle', '[Hz]')
        pt.options(var_out, 'colormap', 'turbo')
        pt.options(var_out, 'zrange', [1e-6, 1e3])
        pt.options(var_out, 'yrange', [1E-2, 32])   # Nyquist 周波数は 32 Hz

In [ ]:
# 2. タイムスパンを変えつつプロット／保存
#time_windows_1 = [
#    np.datetime64('2022-09-01T22:25') + np.timedelta64(5, 'm')*n for n in range(12)
#]
#time_windows_2 = [
#    np.datetime64('2022-09-01T21:30') + np.timedelta64(5, 'm')*n for n in range(6)
#]
#
## 両方のリストを結合してDatetimeIndexに変換
#time_windows = pd.DatetimeIndex(time_windows_1 + time_windows_2)
#
#print(f"Time windows: {time_windows}")
#
#for t0 in time_windows:
#    start_str = str(t0)             # "2022-09-01T22:30:00" など
#    # tplot のタイムスパン設定
#    pt.timespan(start_str, 5, keyword='minute')
#    print(f"plotting window: {start_str}")
#
#    vars_cwt = [
#        'E_fac_x_BS_cwt', 'E_fac_y_BS_cwt', 'E_fac_z_BS_cwt',
#        'B_fac_x_BS_cwt', 'B_fac_y_BS_cwt', 'B_fac_z_BS_cwt'
#    ]
#
#    if os.path.isdir(path_base_save_plot):
#        # フォルダがあれば保存のみ
#        # ファイル名に時刻を埋め込む（コロンがあるとダメなので置換）
#        fn_time = start_str.replace(':', '').replace('T', '_')
#        save_png = os.path.join(path_base_save_plot,
#                                f'EB_fields_fac_cwt_{fn_time}.png')
#
#        fig, axes = pt.tplot(
#            vars_cwt,
#            display=False,
#            return_plot_objects=True,
#            save_png=save_png
#        )
#        plt.close(fig)
#
#    else:
#        # フォルダがなければ表示のみ
#        pt.tplot(
#            vars_cwt,
#            display=True
#        )

# 各成分のNoise(Median)を抽出

In [ ]:
import os
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import pytplot as pt


# --- 1. PSD 変数リストと時間範囲 -----------------------------
vars_psd = [
    'E_fac_x_BS_cwt', 'E_fac_y_BS_cwt', 'E_fac_z_BS_cwt',
    'B_fac_x_BS_cwt', 'B_fac_y_BS_cwt', 'B_fac_z_BS_cwt'
]
t0, t1 = '2022-09-01T21:30:00', '2022-09-01T22:00:00'

# --- 2. 各変数ごとに median を計算して Dataset にまとめる ----
noise_da_dict = {}
for v in vars_psd:
    if v not in pt.data_quants:
        print(f'skip (no {v})')
        continue

    dq_cut = pt.data_quants[v].sel(time=slice(t0, t1))
    freq   = dq_cut.spec_bins.values
    med    = np.nanmedian(dq_cut.data, axis=0)

    med_da = xr.DataArray(
        data   = med,
        dims   = ['frequency'],
        coords = {'frequency': freq},
        name   = v
    )
    noise_da_dict[v] = med_da

noise_ds = xr.Dataset(noise_da_dict)
print(noise_ds)

# --- 3. プロット or 保存 --------------------------------------
fig, ax = plt.subplots(figsize=(6,4))
for key in noise_ds.data_vars:
    label = key.split('_')[0] + '_' + key.split('_')[2]
    ax.loglog(noise_ds['frequency'], noise_ds[key], label=label)

ax.set_xlabel('Frequency [Hz]')
ax.set_ylabel('Median PSD')
ax.set_title(r'Noise floor 21:30-22:00 (median) (E: (mV/m)$^2$/Hz, B: (nT)$^2$/Hz)')
ax.grid(True, which='both', ls=':')
ax.legend()
ax.set_xlim(0.01, 32)
ax.set_ylim(1E-10, 2E1)
plt.tight_layout()

if os.path.isdir(path_base_save_plot):
    # フォルダが存在 → 保存のみ
    # ファイル名にコロンが入るとまずいので除去
    fn = f"noise_median_{t0.replace(':','')}_{t1.replace(':','')}.png"
    save_path = os.path.join(path_base_save_plot, fn)
    fig.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close(fig)
    print(f"saved to {save_path}")
else:
    # フォルダが存在しない → 通常表示
    plt.show()

In [ ]:
import os
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import pytplot as pt

# ------------------------------------------------------------
# 2. clean_vars を pytplot に登録
# ------------------------------------------------------------
vars_psd    = [
    'E_fac_x_BS_cwt', 'E_fac_y_BS_cwt', 'E_fac_z_BS_cwt',
    'B_fac_x_BS_cwt', 'B_fac_y_BS_cwt', 'B_fac_z_BS_cwt'
]
frequency   = noise_ds['frequency'].values  # shape=(Nf,)
clean_vars  = []

for v in vars_psd:
    if v not in pt.data_quants:
        print(f'skip (no {v})')
        continue

    # 元の PSD (time x freq)
    dq        = pt.data_quants[v]
    noise_vec = noise_ds[v].values           # shape=(Nf,)
    cleaned   = dq.data - noise_vec[None, :]
    cleaned[cleaned <= 0] = np.nan           # log 表示できない小数値は NaN に

    # 新変数名
    v_out = v.replace('_cwt', '_clean')
    pt.store_data(
        v_out,
        data={'x': dq.time.values, 'y': cleaned, 'v': frequency}
    )

    # 軸タイトル・凡例・スケール設定
    # 例：ytitle_list に対応する順序でラベルを設定
    ytitle_list     = [
        'E_x (FAC)', 'E_y (FAC)', 'E_z (FAC)',
        'B_x (FAC)', 'B_y (FAC)', 'B_z (FAC)'
    ]
    zsubtitle_list  = [
        '(mV/m)^2/Hz', '(mV/m)^2/Hz', '(mV/m)^2/Hz',
        '(nT)^2/Hz',    '(nT)^2/Hz',    '(nT)^2/Hz'
    ]
    idx = vars_psd.index(v)

    pt.options(v_out, 'ytitle',      ytitle_list[idx])
    pt.options(v_out, 'ysubtitle',   '[Hz]')
    pt.options(v_out, 'ztitle',      'PSD (S–N)')
    pt.options(v_out, 'zsubtitle',   zsubtitle_list[idx])
    pt.options(v_out, 'colormap',    'turbo')
    pt.options(v_out, 'ylog',        1)        # 周波数軸 log
    pt.options(v_out, 'spec',        1)        # スペクトログラムモード
    pt.options(v_out, 'zlog',        1)        # カラー軸 log
    pt.options(v_out, 'zrange',     [1e-6, 1e3])
    pt.options(v_out, 'yrange',     [1E-2, 32]) # Nyquist 周波数まで

    clean_vars.append(v_out)

# ------------------------------------------------------------
# 3. タイムウィンドウを変えつつ保存 or 表示
# ------------------------------------------------------------
#time_windows_1 = [
#    np.datetime64('2022-09-01T22:25') + np.timedelta64(5, 'm')*n for n in range(12)
#]
#time_windows_2 = [
#    np.datetime64('2022-09-01T21:30') + np.timedelta64(5, 'm')*n for n in range(6)
#]
#
## 両方のリストを結合してDatetimeIndexに変換
#time_windows = pd.DatetimeIndex(time_windows_1 + time_windows_2)
#
#for t0 in time_windows:
#    ts = str(t0)  # '2022-09-01T22:30:00'
#    pt.timespan(ts, 5, keyword='minute')
#    print(f"Window: {ts}")
#
#    if os.path.isdir(path_base_save_plot):
#        # フォルダがある → 保存のみ
#        fn_time = ts.replace(':', '').replace('T', '_')
#        save_png = os.path.join(
#            path_base_save_plot,
#            f'EB_fields_fac_cwt_clean_{fn_time}.png'
#        )
#
#        # display=False で plt.show() を抑制、return_plot_objects=True で fig, axes を取得
#        fig, axes = pt.tplot(
#            clean_vars,
#            display=False,             # tplot の出力そのままキャプチャ
#            return_plot_objects=True,
#            save_png=save_png
#        )
#        # 図を閉じて Notebook 上への自動表示も抑制
#        plt.close(fig)
#        print(f"Saved: {save_png}")
#
#    else:
#        # フォルダがない → Notebook 上に表示のみ
#        pt.tplot(
#            clean_vars,
#            display=True
#        )


In [ ]:
import os
import numpy as np
import pytplot as pt
import xarray as xr
import matplotlib.pyplot as plt


# --- 1. B_total を計算し、ローリング平均用の win_pts を定義 ---
Bx = pt.data_quants['B_fac_x'].sel(time=slice(*time_range_T))
By = pt.data_quants['B_fac_y'].sel(time=slice(*time_range_T))
Bz = pt.data_quants['B_fac_z'].sel(time=slice(*time_range_T))

# numpy 配列ではなく xarray.DataArray のまま計算
B_total = np.sqrt(Bx.data**2 + By.data**2 + Bz.data**2) * 1e-9  # nT→T

# 各サンプル間隔を秒で取得
dt = (Bx.time.values[1] - Bx.time.values[0]).astype('timedelta64[ns]').astype(float) * 1e-9

# 例：100秒を移動平均したいなら
win_secs = 100.0
win_pts = int(win_secs / dt)

# xarray.DataArray に戻してローリング平均
B_total_da = xr.DataArray(
    data=B_total,
    coords={'time': Bx.time.values},
    dims=['time']
)
B_total_roll = B_total_da.rolling(time=win_pts, center=True).mean()    #平均しない

# --- 2. pytplot 変数に登録 ----------------------------------
pt.store_data(
    'B_total_T',
    data={
        'x': B_total_roll.time.values,
        'y': B_total_roll.values
    }
)
pt.options('B_total_T', 'ytitle', 'B (total)')
pt.options('B_total_T', 'ysubtitle', '[T]')

# --- 3. プロット or 保存 ----------------------------------
pt.timespan('2022-09-01/22:25:00', 1, keyword='hour')

if os.path.isdir(path_base_save_plot):
    save_path = os.path.join(path_base_save_plot, 'B_total_rolling.png')
    fig, axes = pt.tplot(
        'B_total_T',
        display=False,
        return_plot_objects=True,
        save_png=save_path
    )
    plt.close(fig)
    print(f"Saved plot to {save_path}")
else:
    pt.tplot('B_total_T', display=True)

# 軌道データから、衛星速度(DSI)を導出

In [ ]:
import os
import pyspedas as psp
import pytplot as pt
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

# ──────────────────────────
# 1. 軌道＆磁場ロード＋GSE速度計算
# ──────────────────────────
psp.erg.orb(trange=time_range, level='l2', datatype='def')
psp.erg.mgf(trange=time_range, level='l2', datatype='64hz', coord='gse')

pos   = pt.data_quants['erg_orb_l2_pos_gse'].data   # (Nt,3)[R_E]
t_pos = pt.data_quants['erg_orb_l2_pos_gse'].time.values

R_E_m = 6.378137e6
t_s   = t_pos.astype('datetime64[ns]').astype(float) * 1e-9
v_gse = np.gradient(pos * R_E_m, t_s, axis=0)

v_sc_gse = xr.DataArray(
    data=v_gse, dims=('time','v_dim'),
    coords={'time': t_pos},
    attrs={'units':'m/s','desc':'$V_{sc}$ (GSE)'}
)

pt.store_data('v_sc_gse', data={'x': t_s, 'y': v_gse})
pt.options('v_sc_gse', 'ytitle', r'$V_{sc}$ (GSE) [m/s]')
pt.options('v_sc_gse', 'legend_names', ['x','y','z'])

# ──────────────────────────
# 2. GSE→J2000→DSI 変換
# ──────────────────────────
# (1) GSE→J2000
psp.cotrans(
    name_in   = 'v_sc_gse',
    name_out  = 'v_sc_j2000',
    coord_in  = 'gse',
    coord_out = 'j2000'
)
pt.options('v_sc_j2000', 'ytitle', r'$V_{sc}$ (J2000) [m/s]')
pt.options('v_sc_j2000', 'legend_names', ['x','y','z'])

# (2) J2000→DSI
from pyspedas.projects.erg.satellite.erg.common.cotrans.dsi2j2000 import dsi2j2000

dsi2j2000(name_in='v_sc_j2000', name_out='v_sc_dsi', J20002DSI=True)
pt.options('v_sc_dsi', 'ytitle', r'$V_{sc}$ (DSI) [m/s]')
pt.options('v_sc_dsi', 'legend_names', ['x','y','z'])

# ──────────────────────────
# 3. 各プロットを「保存のみ／表示のみ」で切り替え
# ──────────────────────────
pt.timespan('2022-09-01/22:25:00', 1, keyword='hour')

for var in ['v_sc_gse','v_sc_j2000','v_sc_dsi']:
    if os.path.isdir(path_base_save_plot):
        png_path = os.path.join(path_base_save_plot, f'{var}.png')
        pt.tplot(var, save_png=png_path, display=False)
        plt.close('all')
        print(f"Saved: {png_path}")
    else:
        pt.tplot(var, display=True)

In [ ]:
V_sc_dsi = pt.data_quants['v_sc_dsi']

V_sc_dsi = V_sc_dsi.interp(time=pt.data_quants['B_fac_x_BS'].time, method='linear')

V_sc_fac_np = np.einsum('tji,tj->ti', Rotation_tensor, V_sc_dsi.values)  # (N,3) × (N,3,3) → (N,3)

V_sc_fac_perp_np = np.sqrt(V_sc_fac_np[:, 0]**2 + V_sc_fac_np[:, 1]**2)  # (N,)

pt.store_data('v_sc_fac',
              data={'x': V_sc_dsi.time.values, 'y': V_sc_fac_np},
              attr_dict=V_sc_dsi.attrs)

pt.store_data('v_sc_fac_perp',
              data={'x': V_sc_dsi.time.values, 'y': V_sc_fac_perp_np})

pt.options('v_sc_fac', 'char_size', 15)
pt.options('v_sc_fac', 'ytitle', r'$V_{\mathrm{sc}}$ (FAC)')
pt.options('v_sc_fac', 'ysubtitle', r'[m/s]')
pt.options('v_sc_fac', 'legend_names', [r'$x$ (FAC)', r'$y$ (FAC)', r'$z$ (FAC)'])

pt.options('v_sc_fac_perp', 'ytitle', r'$V_{\mathrm{sc}\perp}$ (FAC)')
pt.options('v_sc_fac_perp', 'char_size', 15)
pt.options('v_sc_fac_perp', 'ysubtitle', r'[m/s]')

vars_to_plot = ['v_sc_fac', 'v_sc_fac_perp']

if os.path.isdir(path_base_save_plot):
    # フォルダがある → 表示せずに PNG で保存のみ
    save_png = os.path.join(path_base_save_plot, 'v_sc_fac_and_perp.png')
    pt.tplot(
        vars_to_plot,
        display=False,      # Notebook 上での自動表示を抑制
        save_png=save_png   # 保存先ファイル名
    )
    # 念のため閉じて自動描画も防止
    plt.close('all')
    print(f"Saved plot to {save_png}")

else:
    # フォルダがなければ通常表示のみ
    pt.tplot(
        vars_to_plot,
        display=True        # 省略可（デフォルトは True）
    )

# イオン流速($\approx$ MHD流速)、イオン温度→イオン熱速度、電子温度→ion acoustic speedの導出

In [ ]:
import pyspedas as psp
import pytplot as pt

psp.erg.lepe(trange=time_range, datatype='3dflux', level='l2', no_update=True)
psp.erg.lepi(trange=time_range, datatype='3dflux', level='l2', no_update=True)
psp.erg.orb(trange=time_range, level='l2', datatype='def', no_update=True)

psp.projects.erg.erg_lep_part_products(
    'erg_lepe_l2_3dflux_FEDU',
    outputs=['moments'],
    mag_name='erg_mgf_l2_mag_64hz_dsi'
)

psp.projects.erg.erg_lep_part_products(
    'erg_lepi_l2_3dflux_FPDU',
    outputs=['moments'],
    mag_name='erg_mgf_l2_mag_64hz_dsi'
)

psp.projects.erg.erg_lep_part_products(
    'erg_lepi_l2_3dflux_FHEDU',
    outputs=['moments'],
    mag_name='erg_mgf_l2_mag_64hz_dsi'
)

psp.projects.erg.erg_lep_part_products(
    'erg_lepi_l2_3dflux_FODU',
    outputs=['moments'],
    mag_name='erg_mgf_l2_mag_64hz_dsi'
)

pt.tplot_math.split_vec('erg_lepi_l2_3dflux_FPDU_velocity')
pt.tplot_math.split_vec('erg_lepi_l2_3dflux_FHEDU_velocity')
pt.tplot_math.split_vec('erg_lepi_l2_3dflux_FODU_velocity')

## ion compositionで平均化

In [ ]:
n_H = pt.data_quants['erg_lepi_l2_3dflux_FPDU_density'].fillna(0)
n_He = pt.data_quants['erg_lepi_l2_3dflux_FHEDU_density'].fillna(0)
n_O = pt.data_quants['erg_lepi_l2_3dflux_FODU_density'].fillna(0)

T_H = pt.data_quants['erg_lepi_l2_3dflux_FPDU_avgtemp'].fillna(0)
T_He = pt.data_quants['erg_lepi_l2_3dflux_FHEDU_avgtemp'].fillna(0)
T_O = pt.data_quants['erg_lepi_l2_3dflux_FODU_avgtemp'].fillna(0)

Vx_H = pt.data_quants['erg_lepi_l2_3dflux_FPDU_velocity_x'].fillna(0)
Vx_He = pt.data_quants['erg_lepi_l2_3dflux_FHEDU_velocity_x'].fillna(0)
Vx_O = pt.data_quants['erg_lepi_l2_3dflux_FODU_velocity_x'].fillna(0)

Vy_H = pt.data_quants['erg_lepi_l2_3dflux_FPDU_velocity_y'].fillna(0)
Vy_He = pt.data_quants['erg_lepi_l2_3dflux_FHEDU_velocity_y'].fillna(0)
Vy_O = pt.data_quants['erg_lepi_l2_3dflux_FODU_velocity_y'].fillna(0)

Vz_H = pt.data_quants['erg_lepi_l2_3dflux_FPDU_velocity_z'].fillna(0)
Vz_He = pt.data_quants['erg_lepi_l2_3dflux_FHEDU_velocity_z'].fillna(0)
Vz_O = pt.data_quants['erg_lepi_l2_3dflux_FODU_velocity_z'].fillna(0)

n_total = n_H + n_He + n_O  # [cm^-3]
Pressure_total = n_H*T_H + n_He*T_He + n_O*T_O  # [eV/cm^3]
Flux_x = n_H*Vx_H + n_He*Vx_He + n_O*Vx_O  # [km/s * cm^-3]
Flux_y = n_H*Vy_H + n_He*Vy_He + n_O*Vy_O
Flux_z = n_H*Vz_H + n_He*Vz_He + n_O*Vz_O

Temperature_avg = Pressure_total / n_total  # [eV]
Vx_avg = Flux_x / n_total  # [km/s]
Vy_avg = Flux_y / n_total
Vz_avg = Flux_z / n_total

proton_mass_kg = 1.6726219e-27  # kg
He_mass_kg = proton_mass_kg * 4
O_mass_kg = proton_mass_kg * 16

ion_mass_avg = (n_H*proton_mass_kg + n_He*He_mass_kg + n_O*O_mass_kg) / n_total  # [kg]

pt.store_data('erg_lepi_mass_avg', data={'x': n_total.time.values, 'y': ion_mass_avg.values})
pt.options('erg_lepi_mass_avg', 'ytitle', 'Average Ion Mass')
pt.options('erg_lepi_mass_avg', 'ysubtitle', '[kg]')

pt.store_data('erg_lepi_mass_avg_per_proton', data={'x': n_total.time.values, 'y': ion_mass_avg.values/proton_mass_kg})
pt.options('erg_lepi_mass_avg_per_proton', 'ytitle', 'Average Ion Mass')
pt.options('erg_lepi_mass_avg_per_proton', 'ysubtitle', '[proton mass]')

pt.store_data('erg_lepi_temperature_avg', data={'x': Temperature_avg.time.values, 'y': Temperature_avg.values})
pt.options('erg_lepi_temperature_avg', 'ytitle', 'Average Ion Temperature')
pt.options('erg_lepi_temperature_avg', 'ysubtitle', '[eV]')

pt.store_data('erg_lepi_velocity_x_avg', data={'x': Vx_avg.time.values, 'y': Vx_avg.values})
pt.options('erg_lepi_velocity_x_avg', 'ytitle', 'Average Ion Velocity (DSI x)')
pt.options('erg_lepi_velocity_x_avg', 'ysubtitle', '[km/s]')

pt.store_data('erg_lepi_velocity_y_avg', data={'x': Vy_avg.time.values, 'y': Vy_avg.values})
pt.options('erg_lepi_velocity_y_avg', 'ytitle', 'Average Ion Velocity (DSI y)')
pt.options('erg_lepi_velocity_y_avg', 'ysubtitle', '[km/s]')

pt.store_data('erg_lepi_velocity_z_avg', data={'x': Vz_avg.time.values, 'y': Vz_avg.values})
pt.options('erg_lepi_velocity_z_avg', 'ytitle', 'Average Ion Velocity (DSI z)')
pt.options('erg_lepi_velocity_z_avg', 'ysubtitle', '[km/s]')

In [ ]:
var = ['erg_lepi_temperature_avg', 'erg_lepe_l2_3dflux_FEDU_avgtemp']

pt.options('erg_lepe_l2_3dflux_FEDU_avgtemp', 'ytitle', r'$\mathrm{e}^{-}$ Temperature')
pt.options('erg_lepi_temperature_avg', 'ytitle', r'ion Temperature')
pt.options(var, 'ysubtitle', '[eV]')
pt.options(var, 'yrange', [1E2, 1E4])
pt.options(var, 'ylog', True)

if os.path.isdir(path_base_save_plot):
    # フォルダが存在 → 保存のみ
    fn = os.path.join(path_base_save_plot, f'plasma_temperature.png')
    pt.tplot(
        var,
        display=False,    # Notebook 上の自動表示を抑制
        save_png=fn       # PNG をこのファイル名で保存
    )
    plt.close('all')     # 完全に閉じて自動表示も防止
    print(f"Saved plot to {fn}")
else:
    # フォルダがなければ通常表示のみ
    pt.tplot(var, display=True)

In [ ]:
var = ['erg_lepi_mass_avg_per_proton']
pt.options(var, 'yrange', [5e-1, 3e0])

if os.path.isdir(path_base_save_plot):
    # フォルダが存在 → 保存のみ
    fn = os.path.join(path_base_save_plot, f'ion_mass.png')
    pt.tplot(
        var,
        display=False,    # Notebook 上の自動表示を抑制
        save_png=fn       # PNG をこのファイル名で保存
    )
    plt.close('all')     # 完全に閉じて自動表示も防止
    print(f"Saved plot to {fn}")
else:
    # フォルダがなければ通常表示のみ
    pt.tplot(var, display=True)

In [ ]:
import xarray as xr
import numpy as np
import pytplot as pt

Temp_electron = pt.data_quants['erg_lepe_l2_3dflux_FEDU_avgtemp']
dt_e = (Temp_electron.time[1] - Temp_electron.time[0]).astype('timedelta64[ns]').astype(float) * 1E-9
win_pts_e = int(100.0 / dt_e)
Temp_electron = Temp_electron.rolling(time=win_pts_e, center=True).mean('time')
Temp_electron_interp = Temp_electron.interp(time=pt.data_quants['B_fac_x'].time, method='linear')

Temp_ion = pt.data_quants['erg_lepi_temperature_avg']
dt_i = (Temp_ion.time[1] - Temp_ion.time[0]).astype('timedelta64[ns]').astype(float) * 1E-9
win_pts_i = int(100.0 / dt_i)
Temp_ion = Temp_ion.rolling(time=win_pts_i, center=True).mean('time')
Temp_ion_interp = Temp_ion.interp(time=pt.data_quants['B_fac_x'].time, method='linear')

mass_ion = pt.data_quants['erg_lepi_mass_avg']
dt_i = (mass_ion.time[1] - mass_ion.time[0]).astype('timedelta64[ns]').astype(float) * 1E-9
win_pts_i = int(100.0 / dt_i)
mass_ion = mass_ion.rolling(time=win_pts_i, center=True).mean('time')
mass_ion_interp = mass_ion.interp(time=pt.data_quants['B_fac_x'].time, method='linear')

mass_proton_kg = 1.6726219E-27
elementary_charge = 1.60218e-19  # C

B_total = pt.data_quants['B_total_T'].rolling(time=win_pts_i, center=True).mean('time')
B_total_interp = B_total.interp(time=pt.data_quants['B_fac_x'].time, method='linear')

ion_cyclo_freq = B_total_interp * elementary_charge / (2 * np.pi * mass_proton_kg)  # Hz

V_th_ion = np.sqrt(2 * Temp_ion_interp.values * elementary_charge / mass_ion_interp.values) * 1E-3  # km/s
C_s_ion = np.sqrt(Temp_electron_interp.values * elementary_charge / mass_ion_interp.values) * 1E-3  # km/s

pt.store_data('V_th_ion', data={'x': Temp_ion_interp.time.values, 'y': V_th_ion})
pt.options('V_th_ion', 'ytitle', r'$V_{\mathrm{th, i}}$')
pt.options('V_th_ion', 'ysubtitle', '[km/s]')

pt.store_data('C_s_ion', data={'x': Temp_ion_interp.time.values, 'y': C_s_ion})
pt.options('C_s_ion', 'ytitle', r'$C_{\mathrm{s}}$')
pt.options('C_s_ion', 'ysubtitle', '[km/s]')

pt.store_data('ion_cyclo_freq', data={'x': B_total_interp.time.values, 'y': ion_cyclo_freq.values})
pt.options('ion_cyclo_freq', 'ytitle', r'$f_{\mathrm{ci}}$')
pt.options('ion_cyclo_freq', 'ysubtitle', '[Hz]')

var = ['V_th_ion', 'C_s_ion', 'ion_cyclo_freq']

if os.path.isdir(path_base_save_plot):
    # フォルダがある → Jupyter 上に表示せず PNG で保存のみ
    save_png = os.path.join(path_base_save_plot, 'parameter_1.png')
    pt.tplot(
        var,
        display=False,   # Notebook 上の自動表示を抑制
        save_png=save_png
    )
    plt.close('all')    # 全ての Figure を閉じて描画も防止
    print(f"Saved plot to {save_png}")
else:
    # フォルダがない → Notebook 上に表示のみ
    pt.tplot(var, display=True)

In [ ]:
import xarray as xr
import numpy as np
import pytplot as pt

pt.store_data('erg_lepi_velocity', data={'x': pt.data_quants['erg_lepi_velocity_x_avg'].time.values, 'y': np.vstack((
    pt.data_quants['erg_lepi_velocity_x_avg'].values,
    pt.data_quants['erg_lepi_velocity_y_avg'].values,
    pt.data_quants['erg_lepi_velocity_z_avg'].values
)).T})

V_ion_dsi = pt.data_quants['erg_lepi_velocity']

V_ion_dsi_interp = V_ion_dsi.interp(time=pt.data_quants['B_fac_x'].time, method='linear')

print(Rotation_tensor.shape)  # (N,3,3)
print(V_ion_dsi_interp.shape)  # (N,3)

V_ion_fac_np = np.einsum('tji,tj->ti', Rotation_tensor, V_ion_dsi_interp.values)  # (N,3) × (N,3,3) → (N,3)

V_ion_fac_np_perp = np.sqrt(V_ion_fac_np[:, 0]**2 + V_ion_fac_np[:, 1]**2)  # (N,)

pt.store_data('erg_lepi_velocity_fac', data={'x': V_ion_dsi_interp.time.values, 'y': V_ion_fac_np}, attr_dict=V_ion_dsi_interp.attrs)

pt.store_data('erg_lepi_velocity_fac_perp', data={'x': V_ion_dsi_interp.time.values, 'y': V_ion_fac_np_perp})

pt.tplot_math.split_vec('erg_lepi_velocity_fac')

pt.options('erg_lepi_velocity_fac_x', 'ytitle', r'$V_{\mathrm{iflow}x}$')
pt.options('erg_lepi_velocity_fac_y', 'ytitle', r'$V_{\mathrm{iflow}y}$')
pt.options('erg_lepi_velocity_fac_z', 'ytitle', r'$V_{\mathrm{iflow}z}$')
pt.options(['erg_lepi_velocity_fac_x', 'erg_lepi_velocity_fac_y', 'erg_lepi_velocity_fac_z'], 'ysubtitle', '(FAC) [km/s]')
pt.options(['erg_lepi_velocity_fac_x', 'erg_lepi_velocity_fac_y', 'erg_lepi_velocity_fac_z'], 'legend_names', None)

pt.options('erg_lepi_velocity_fac_perp', 'ytitle', r'$V_{\mathrm{iflow}\perp}$')
pt.options('erg_lepi_velocity_fac_perp', 'ysubtitle', '(FAC) [km/s]')
pt.options('erg_lepi_velocity_fac_perp', 'char_size', 15)

vars_fac = [
    'erg_lepi_velocity_fac_x', 'erg_lepi_velocity_fac_y', 'erg_lepi_velocity_fac_z',
    'erg_lepi_velocity_fac_perp'
]

if os.path.isdir(path_base_save_plot):
    # フォルダがある → Jupyter 上に表示せず PNG で保存のみ
    save_png = os.path.join(path_base_save_plot, 'V_iflow_fac.png')
    pt.tplot(
        vars_fac,
        display=False,   # Notebook 上の自動表示を抑制
        save_png=save_png
    )
    plt.close('all')    # 全ての Figure を閉じて描画も防止
    print(f"Saved plot to {save_png}")
else:
    # フォルダがない → Notebook 上に表示のみ
    pt.tplot(vars_fac, display=True)

In [ ]:
V_sys_fac_np_perp = np.sqrt((V_ion_fac_np[:, 0]-V_sc_fac_np[:, 0]*1E-3)**2 + (V_ion_fac_np[:, 1]-V_sc_fac_np[:, 1]*1E-3)**2)

pt.store_data('v_sys_fac_perp',
              data={'x': V_sc_dsi.time.values, 'y': V_sys_fac_np_perp})

pt.store_data('v_sys_fac',
              data={'x': V_sc_dsi.time.values, 'y': V_ion_fac_np-V_sc_fac_np*1E-3},
              attr_dict=V_sc_dsi.attrs)

pt.options('v_sys_fac', 'char_size', 15)
pt.options('v_sys_fac', 'ytitle', r'$V_{\mathrm{sys}}$ (FAC)')
pt.options('v_sys_fac', 'ysubtitle', r'[km/s]')
pt.options('v_sys_fac', 'legend_names', [r'$x$ (FAC)', r'$y$ (FAC)', r'$z$ (FAC)'])

pt.options('v_sys_fac_perp', 'ytitle', r'$V_{\mathrm{sys}\perp}$ (FAC)')
pt.options('v_sys_fac_perp', 'char_size', 15)
pt.options('v_sys_fac_perp', 'ysubtitle', r'[km/s]')

vars_fac = ['v_sys_fac', 'v_sys_fac_perp']

if os.path.isdir(path_base_save_plot):
    # フォルダがある → Jupyter 上に表示せず PNG で保存のみ
    save_png = os.path.join(path_base_save_plot, 'V_sys_fac.png')
    pt.tplot(
        vars_fac,
        display=False,   # Notebook 上の自動表示を抑制
        save_png=save_png
    )
    plt.close('all')    # 全ての Figure を閉じて描画も防止
    print(f"Saved plot to {save_png}")
else:
    # フォルダがない → Notebook 上に表示のみ
    pt.tplot(vars_fac, display=True)

In [ ]:
import xarray as xr
import numpy as np
import pytplot as pt

Vperp_rolling = pt.data_quants['v_sys_fac_perp']
dt_i = (Vperp_rolling.time[1] - Vperp_rolling.time[0]).astype('timedelta64[ns]').astype(float) * 1E-9
win_pts_i = int(100.0 / dt_i)
Vperp_rolling = Vperp_rolling.rolling(time=win_pts_i, center=True).mean('time')

pt.store_data('v_sys_fac_perp_rolling',
              data={'x': Vperp_rolling.time.values, 'y': Vperp_rolling},
              attr_dict=Vperp_rolling.attrs)


vars_fac = 'v_sys_fac_perp_rolling'

if os.path.isdir(path_base_save_plot):
    # フォルダがある → Jupyter 上に表示せず PNG で保存のみ
    save_png = os.path.join(path_base_save_plot, 'V_sys_fac_perp_rolling.png')
    pt.tplot(
        vars_fac,
        display=False,   # Notebook 上の自動表示を抑制
        save_png=save_png
    )
    plt.close('all')    # 全ての Figure を閉じて描画も防止
    print(f"Saved plot to {save_png}")
else:
    # フォルダがない → Notebook 上に表示のみ
    pt.tplot(vars_fac, display=True)

## 電子数密度の導出 (UHR, LEP-e)

In [ ]:
import xarray as xr
import numpy as np
import pytplot as pt
import pyspedas as psp

n_electron_lepe_interp = pt.data_quants['erg_lepe_l2_3dflux_FEDU_density'].interp(time=pt.data_quants['B_fac_x'].time, method='linear')

psp.erg.pwe_hfa(trange=time_range, level='l3', no_update=True)

n_electron_hfa_interp = pt.data_quants['erg_pwe_hfa_l3_1min_ne_mgf'].interp(time=pt.data_quants['B_fac_x'].time, method='linear')

n_electron_mid_interp = (n_electron_lepe_interp + n_electron_hfa_interp) / 2.0

pt.store_data('n_electron_mid_interp',
              data={'x': n_electron_mid_interp.time.values, 'y': n_electron_mid_interp.values})
pt.options('n_electron_mid_interp', 'ytitle', r'$n_{e}$ (mid)')
pt.options('n_electron_mid_interp', 'ysubtitle', r'[cm$^{-3}$]')
pt.options('n_electron_mid_interp', 'ylog', True)

pt.store_data('n_electron_lepe_interp',
              data={'x': n_electron_lepe_interp.time.values, 'y': n_electron_lepe_interp.values})
pt.options('n_electron_lepe_interp', 'ytitle', r'$n_{e}$ (LEP-e)')
pt.options('n_electron_lepe_interp', 'ysubtitle', r'[cm$^{-3}$]')
pt.options('n_electron_lepe_interp', 'ylog', True)

pt.store_data('n_electron_hfa_interp',
              data={'x': n_electron_hfa_interp.time.values, 'y': n_electron_hfa_interp.values})
pt.options('n_electron_hfa_interp', 'ytitle', r'$n_{e}$ (PWE-HFA)')
pt.options('n_electron_hfa_interp', 'ysubtitle', r'[cm$^{-3}$]')
pt.options('n_electron_hfa_interp', 'ylog', True)

vars_fac = ['n_electron_hfa_interp', 'n_electron_mid_interp', 'n_electron_lepe_interp']


if os.path.isdir(path_base_save_plot):
    save_path = os.path.join(path_base_save_plot, 'n_electron_comparison.png')
    fig, axes = pt.tplot(
        vars_fac,
        display=False,
        return_plot_objects=True,
        save_png=save_path
    )
    plt.close(fig)
    print(f"Saved plot to {save_path}")
else:
    pt.tplot(vars_fac, display=True)


## Alfvén speedの導出

In [ ]:
import xarray as xr
import numpy as np
import pytplot as pt

dt_e = (n_electron_lepe_interp.time[1] - n_electron_lepe_interp.time[0]).astype('timedelta64[ns]').astype(float) * 1E-9
win_pts_e = int(100.0 / dt_e)
n_electron_lepe_interp = n_electron_lepe_interp.rolling(time=win_pts_e, center=True).mean('time')
n_electron_hfa_interp = n_electron_hfa_interp.rolling(time=win_pts_e, center=True).mean('time')
n_electron_mid_interp = n_electron_mid_interp.rolling(time=win_pts_e, center=True).mean('time')

mu_0 = 4 * np.pi * 1E-7  # H/m

v_A_lepe = B_total_interp.values / np.sqrt(mu_0 * n_electron_lepe_interp.values * 1E6 * mass_ion_interp.values) * 1E-3  # km/s
v_A_hfa = B_total_interp.values / np.sqrt(mu_0 * n_electron_hfa_interp.values * 1E6 * mass_ion_interp.values) * 1E-3  # km/s
v_A_mid = B_total_interp.values / np.sqrt(mu_0 * n_electron_mid_interp.values * 1E6 * mass_ion_interp.values) * 1E-3  # km/s

pt.store_data('v_A_lepe', data={'x': n_electron_lepe_interp.time.values, 'y': v_A_lepe})
pt.options('v_A_lepe', 'ytitle', r'$v_{\mathrm{A}}$ (LEP-e)')
pt.options('v_A_lepe', 'ysubtitle', '[km/s]')

pt.store_data('v_A_hfa', data={'x': n_electron_hfa_interp.time.values, 'y': v_A_hfa})
pt.options('v_A_hfa', 'ytitle', r'$v_{\mathrm{A}}$ (PWE-HFA)')
pt.options('v_A_hfa', 'ysubtitle', '[km/s]')

pt.store_data('v_A_mid', data={'x': n_electron_mid_interp.time.values, 'y': v_A_mid})
pt.options('v_A_mid', 'ytitle', r'$v_{\mathrm{A}}$ (mid)')
pt.options('v_A_mid', 'ysubtitle', '[km/s]')

vars_fac = ['v_A_hfa', 'v_A_mid', 'v_A_lepe']

if os.path.isdir(path_base_save_plot):
    # フォルダがある → Jupyter 上に表示せず PNG で保存のみ
    save_png = os.path.join(path_base_save_plot, 'v_A_comparison.png')
    fig, axes = pt.tplot(
        vars_fac,
        display=False,   # Notebook 上の自動表示を抑制
        return_plot_objects=True,
        save_png=save_png
    )
    plt.close(fig)    # 全ての Figure を閉じて描画も防止
    print(f"Saved plot to {save_png}")
else:
    # フォルダがない → Notebook 上に表示のみ
    pt.tplot(vars_fac, display=True)

## $\beta_\mathrm{i}$, $\tau := T_{\mathrm{i}} / T_{\mathrm{e}}$の導出

In [ ]:
import pytplot as pt
import xarray as xr

beta_i_lepe = (V_th_ion**2) / (v_A_lepe**2)
beta_i_hfa = (V_th_ion**2) / (v_A_hfa**2)
beta_i_mid = (V_th_ion**2) / (v_A_mid**2)

tau = Temp_ion_interp.values / Temp_electron_interp.values

pt.store_data('beta_i_lepe', data={'x': Temp_ion_interp.time.values, 'y': beta_i_lepe})
pt.options('beta_i_lepe', 'ytitle', r'$\beta_{i}$ (LEP-e)')
pt.options('beta_i_lepe', 'ylog', True)

pt.store_data('beta_i_hfa', data={'x': Temp_ion_interp.time.values, 'y': beta_i_hfa})
pt.options('beta_i_hfa', 'ytitle', r'$\beta_{i}$ (PWE-HFA)')
pt.options('beta_i_hfa', 'ylog', True)

pt.store_data('beta_i_mid', data={'x': Temp_ion_interp.time.values, 'y': beta_i_mid})
pt.options('beta_i_mid', 'ytitle', r'$\beta_{i}$ (mid)')
pt.options('beta_i_mid', 'ylog', True)

pt.store_data('beta_i', data=['beta_i_hfa', 'beta_i_mid', 'beta_i_lepe'])
pt.options('beta_i', 'legend_names', ['PWE-HFA', 'mid', 'LEP-e'])
pt.options('beta_i', 'line_color', ['blue', 'green', 'red'])
pt.options('beta_i', 'ytitle', r'$\beta_{i}$')
pt.options('beta_i', 'ylog', True)
pt.options('beta_i', 'yrange', [1E-4, 1E-1])

pt.store_data('tau', data={'x': Temp_ion_interp.time.values, 'y': tau})
pt.options('tau', 'ytitle', r'$\tau$')

vars_fac = ['beta_i', 'tau']

if os.path.isdir(path_base_save_plot):
    # フォルダがある → Jupyter 上に表示せず PNG で保存のみ
    save_png = os.path.join(path_base_save_plot, 'parameter_2.png')
    pt.tplot(
        vars_fac,
        display=False,   # Notebook 上の自動表示を抑制
        save_png=save_png
    )
    plt.close('all')    # 全ての Figure を閉じて描画も防止
    print(f"Saved plot to {save_png}")
else:
    # フォルダがない → Notebook 上に表示のみ
    pt.tplot(vars_fac, display=True)

# PSDのplot (横軸: frequency [Hz])

In [ ]:
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
import os
import matplotlib as mpl
import sys
sys.path.append("..")
import module_handmade.psd_plotter_ERG_3numden as pp
importlib.reload(pp)

mpl.rcdefaults()
mpl.rcParams['font.size'] = 12

# ------------------------------------------------------------
# 0. 出力フォルダの用意
# ------------------------------------------------------------
out_dir = '/mnt/j/KAW_observation/E_B_ratio_ERG/2022-09-01/22-24_cleaned_CWT_avg_3numben_1sec'
os.makedirs(out_dir, exist_ok=True)

# ------------------------------------------------------------
# 1. データの読み込みと準備（モジュール関数を呼び出す）
# ------------------------------------------------------------
data_dict = pp.load_and_prepare_data(cutoff_freq=[1/100, 1/8, 4/8, 32])

if data_dict is None:
    print("データ準備に失敗したため、処理を終了します。")
    exit()

# ------------------------------------------------------------
# 2. 解析対象の時間を決め、1秒ごとにループ
# ------------------------------------------------------------

def process_and_save_plot(t_start, data_dict, out_dir):
    """
    指定された単一の時刻について、スペクトルをプロットし、画像を保存する関数。
    """
    # モジュール関数を呼び出してプロットを作成
    # psd_plotter を pp としてインポートしている前提
    fig = pp.plot_freq_spectrum(t_start, data_dict, interval_sec=1)
    
    # figがNoneでなければ（データがあってプロットが作成されれば）保存
    if fig is not None:
        try:
            # ファイル名に使いやすいように文字列に変換
            fname = pd.Timestamp(t_start).strftime('%Y-%m-%dT%H%M%S.png')
            fig.savefig(os.path.join(out_dir, fname), dpi=200)
        finally:
            # 保存に失敗しても、メモリ解放のために必ずクローズする
            plt.close(fig)

time_range = ['2022-09-01T22:25:00', '2022-09-01T23:25:00']
t_min, t_max = pd.to_datetime(time_range)

# pandas.date_rangeで1秒ごとのタイムスタンプを生成
time_steps = pd.date_range(start=t_min, end=t_max, freq='1s')

print(f"Processing {len(time_steps)} plots in parallel...")

# forループの代わりにParallelを呼び出す
Parallel(n_jobs=-1, verbose=10)(
    delayed(process_and_save_plot)(t_start, data_dict, out_dir) for t_start in time_steps
)

print('Finished saving all plots!')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import matplotlib as mpl
import sys
import gc # ガベージコレクションをインポート
from joblib import Parallel, delayed # joblibをインポート

sys.path.append("..")
import module_handmade.psd_plotter_ERG_3numden as pp
import importlib

importlib.reload(pp) # モジュールを修正した場合、リロードする

mpl.rcdefaults()
mpl.rcParams['font.size'] = 12

# ------------------------------------------------------------
# 0. 出力フォルダの用意
# ------------------------------------------------------------
out_dir = '/mnt/j/KAW_observation/E_B_ratio_ERG/2022-09-01/22-24_cleaned_CWT_avg_3numben_60sec_k_rhoi_fit_3_10'
os.makedirs(out_dir, exist_ok=True)

# ------------------------------------------------------------
# 1. データの読み込みと準備（ループ前に一度だけ実行）
# ------------------------------------------------------------
print("Loading and preparing data...")
data_dict = pp.load_and_prepare_data(cutoff_freq=[1/100, 1/8, 4/8, 32])

if data_dict is None:
    print("データ準備に失敗したため、処理を終了します。")
    exit()

# ------------------------------------------------------------
# 2. 並列処理のためのラッパー関数を定義
# ------------------------------------------------------------
def process_and_save_k_plot(t_start, data_dict, out_dir, k_range, n_bins, fit_range, dpi=200):
    """
    指定された単一の時刻について、kスペクトルをプロットし、画像を保存する関数。
    """
    fig, fit_results = pp.plot_k_spectrum(
        t_start, 
        data_dict, 
        interval_sec=60, 
        k_range=k_range, 
        n_bins=n_bins,
        fit_range=fit_range
    )
    
    if fig is not None:
        try:
            fname = pd.Timestamp(t_start).strftime('%Y-%m-%dT%H%M%S_k_spec.png')
            fig.savefig(os.path.join(out_dir, fname), dpi=dpi)
        finally:
            plt.close(fig) # メモリ解放
            gc.collect()   # ガベージコレクション
    
    return fit_results

# ------------------------------------------------------------
# 3. 解析対象の時間を決め、並列処理で一気に実行
# ------------------------------------------------------------
time_range = ['2022-09-01T22:25:00', '2022-09-01T23:25:00']
t_min, t_max = pd.to_datetime(time_range)
time_steps = pd.date_range(start=t_min, end=t_max, freq='60s')

m_e = 9.1093837E-31
m_i = 1.67262192E-27
sqrt_m_i_m_e = np.sqrt(m_i / m_e)

# プロットのパラメータ
k_range_to_use      = (1e-1, 1e2)
n_bins_to_use       = 30
fit_range_to_use    = (3, 10)

print(f"Processing {len(time_steps)} plots in parallel...")

# forループの代わりにParallelを呼び出す
results_list = Parallel(n_jobs=-1, verbose=10)(
    delayed(process_and_save_k_plot)(
        t_start, 
        data_dict, 
        out_dir, 
        k_range=k_range_to_use, 
        n_bins=n_bins_to_use,
        fit_range=fit_range_to_use
    ) for t_start in time_steps
)

print('Finished saving all plots!')

# ------------------------------------------------------------
# 4. フィッティング結果の保存と可視化
# ------------------------------------------------------------
print("\n--- Saving and plotting fitting results ---")

# Noneが含まれる可能性を考慮してフィルタリング
results_list = [r for r in results_list if r is not None]

if results_list:
    # リストからDataFrameを作成
    df_results = pd.DataFrame(results_list)
    df_results = df_results.set_index('time').sort_index()

    # CSVファイルとして保存
    time_str_start = t_min.strftime('%Y%m%d_%H%M%S')
    time_str_end = t_max.strftime('%Y%m%d_%H%M%S')
    kappa_csv_filename = f'kappa_timeseries_{time_str_start}_to_{time_str_end}.csv'
    csv_path = os.path.join(out_dir, kappa_csv_filename)
    df_results.to_csv(csv_path)
    print(f"Fitting results saved to {csv_path}")

    # --- 時間変化をプロット ---
    fig_kappa, ax = plt.subplots(figsize=(12, 5))
    
    # kappa_Bのプロット（エラーバー付き）
    ax.errorbar(df_results.index, df_results['kappa_B'], yerr=df_results['kappa_B_err'],
                fmt='o-', color='blue', label=r'$\kappa_B$', ms=4, elinewidth=1, capsize=3)
    
    # kappa_Eのプロット（エラーバー付き）
    ax.errorbar(df_results.index, df_results['kappa_E'], yerr=df_results['kappa_E_err'],
                fmt='o-', color='orange', label=r'$\kappa_E$', ms=4, elinewidth=1, capsize=3)
    
    ax.set_title('Time evolution of spectral index $\kappa$ (ERG)')
    ax.set_ylabel('Spectral index $\kappa$')
    ax.set_xlabel('Time')
    ax.legend()
    ax.minorticks_on()
    ax.grid(True, linestyle=':', which='both')
    
    # プロットを画像として保存
    kappa_plot_filename = f'kappa_timeseries_{time_str_start}_to_{time_str_end}.png'
    kappa_plot_path = os.path.join(out_dir, kappa_plot_filename)
    fig_kappa.savefig(kappa_plot_path, dpi=200)
    plt.close(fig_kappa)
    print(f"Time series plot of kappa saved to {kappa_plot_path}")

else:
    print("No fitting results to save or plot.")